# 03. 같은 강종의 LOT끼리 온도·출력 비교하기

[이슈 #71](https://github.com/posco-3bungi-pdm/pdm-project/issues/71) · 02번에서 LOT별로 본 뒤 읽는 비교임. 01·02번을 먼저 실행하지 않아도 독립 실행됨.

**같은 강종을 묶되 LOT를 평균으로 합치지 않음. 각 LOT의 선을 그대로 겹쳐 봄.**

- 가로축은 **첫 온도 관측 후 경과시간(분)**임. `60`은 첫 관측 후 1시간임. 실제 시계의 01:00을 뜻하지 않음.
- 짧은 LOT를 긴 LOT 길이에 맞춰 늘리지 않음. 기록이 끝난 뒤에는 선이 없음.
- 같은 경과시간이 같은 소재 위치나 같은 조업 단계라는 뜻은 아님.
- 같은 LOT는 모든 그림에서 같은 색임. 진한 선은 1분 평균, 옅은 색은 같은 분의 최소~최대임.
- TC 단위 ℃·OP 단위 %는 데이터사전의 가정임. CP는 단위 미확인임.

`VIEW_GRADE`만 바꿔 화면에서 볼 강종을 선택함. 7개 강종의 그림은 전부 저장함. LOT가 하나뿐인 강종은 **비교 LOT 없음**이라고 표시함.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import font_manager
from IPython.display import display

# 저장소, notebooks, 현재 하위 폴더 중 어디서 실행해도 같은 원자료를 읽기 위함
ROOT = Path.cwd()
if ROOT.name == "타깃_채널_분석_namelesspark":
    ROOT = ROOT.parent.parent
elif ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data"
if not (DATA / "T-CR1-CAL01_온도.csv").exists():
    DATA = DATA / "raw"
FIGURES = ROOT / "figures" / "타깃_채널_비교_namelesspark"

# 설치된 한글 폰트를 쓰기 위함
fonts = {font.name for font in font_manager.fontManager.ttflist}
for font in ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"]:
    if font in fonts:
        plt.rcParams["font.family"] = font
        break
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.spines.top"] = False

TARGETS = ["TC-Z3", "OP-Z3", "TC-Z4", "OP-Z4", "TC-Z5", "OP-Z5", "CP-Z4"]
raw = pd.read_csv(DATA / "T-CR1-CAL01_온도.csv", encoding="utf-8-sig", parse_dates=["MEAS_DT"])
events = pd.read_csv(DATA / "G-02_조업이벤트.csv", encoding="utf-8-sig", parse_dates=["EVT_DT"])
raw = raw.sort_values(["LOT_NO", "MEAS_DT"])

# 이 자료는 LOT 첫 관측과 강종변경 시각이 일치함. 해당 시각으로 강종을 연결함
grade_events = events.loc[events.PLANT_CD.eq("CR1") & events.EVT_TYPE.eq("강종변경")]
lots = raw.groupby("LOT_NO").agg(시작=("MEAS_DT", "min"), 끝=("MEAS_DT", "max"))
lots["강종"] = lots.시작.map(grade_events.set_index("EVT_DT").AFT_VAL)

# LOT마다 따로 집계하므로 서로 다른 LOT 사이의 빈 날짜를 메우지 않음
# OP의 0은 실제 출력임. 결측으로 바꾸지 않음
means, lows, highs, counts = {}, {}, {}, {}
for lot, rows in raw.groupby("LOT_NO"):
    minutes = rows.set_index("MEAS_DT")[TARGETS].resample("1min")
    means[lot] = minutes.mean()
    lows[lot] = minutes.min()
    highs[lot] = minutes.max()
    counts[lot] = minutes.count()

# 아래 2개는 팀에서 확정한 해석임. 전체 LOT를 고장으로 표시하는 뜻은 아님
operation = {240058: "정기수리를 위한 정지", 240061: "정기수리 후 재가동"}


## 1. 비교할 LOT 수 확인하기

LOT가 하나뿐인 SGCC·DP590·SPCD는 강종 내에서 반복되는 현상인지 확인할 수 없음. 같은 강종이라도 운전 상태가 다를 수 있어 LOT를 따로 표시함.


In [ ]:
grade_table = lots.reset_index().groupby("강종", sort=False).agg(
    LOT수=("LOT_NO", "size"),
    LOT목록=("LOT_NO", list),
)
display(grade_table)


## 2. 같은 색의 LOT를 Z3 → Z4 → Z5 → CP 순서로 보기

예를 들어 SPHC에서 061은 재가동 LOT임. 다른 SPHC LOT와 출발 온도가 다른 이유를 먼저 구분하고 읽기. 세 구역 그림은 같은 온도축 범위를 사용하므로 구역마다 다르게 축을 늘려 보이지 않게 함.


In [ ]:
# 화면에서 볼 강종임. 다른 강종도 파일로는 모두 저장함
VIEW_GRADE = "SPHC"
COLORS = ["#2B70B2", "#D17A25", "#448A65", "#9863A5"]

for grade, group in lots.groupby("강종", sort=False):
    lot_ids = group.index.tolist()
    message = "같은 강종의 LOT 비교" if len(lot_ids) > 1 else "비교 LOT 없음"
    legend_labels = []
    for lot in lot_ids:
        label = f"{lot} ({lots.loc[lot, '시작']:%m/%d})"
        if lot in operation:
            label += " · " + operation[lot]
        legend_labels.append(label)

    # Z3·Z4·Z5 그림의 온도축 범위를 통일하기 위함
    tc_min = min(lows[lot][["TC-Z3", "TC-Z4", "TC-Z5"]].min().min() for lot in lot_ids) - 1
    tc_max = max(highs[lot][["TC-Z3", "TC-Z4", "TC-Z5"]].max().max() for lot in lot_ids) + 1

    # 위에는 온도끼리, 아래에는 출력끼리 비교함. 색은 LOT를 구분하기 위함
    for zone in [3, 4, 5]:
        fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True, layout="constrained")
        for index, (lot, label) in enumerate(zip(lot_ids, legend_labels)):
            color = COLORS[index % len(COLORS)]
            mean, low, high = means[lot], lows[lot], highs[lot]
            elapsed = (mean.index - lots.loc[lot, "시작"]).total_seconds() / 60
            for ax, prefix, unit in zip(axes, ["TC", "OP"], ["온도 (℃)", "출력 (%)"]):
                sensor = f"{prefix}-Z{zone}"
                ax.plot(elapsed, mean[sensor], color=color, linewidth=1, label=label)
                ax.fill_between(elapsed, low[sensor], high[sensor], color=color, alpha=0.08)
                ax.set_ylabel(unit)
                ax.grid(alpha=0.15)
        axes[0].set_ylim(tc_min, tc_max)
        axes[1].set_ylim(0, 105)
        axes[0].legend(loc="upper left", fontsize=8)
        axes[1].set_xlabel("첫 온도 관측 후 경과시간 (분) · 선: 1분 평균 · 옅은 색: 그 분의 최소~최대")
        fig.suptitle(f"{grade} | Z{zone} 온도·출력 | {message}")
        output = FIGURES / f"Z{zone}_온도_출력" / "강종별"
        output.mkdir(parents=True, exist_ok=True)
        fig.savefig(output / f"{grade}_Z{zone}_온도_출력.png", dpi=140)
        if grade == VIEW_GRADE:
            plt.show()
        plt.close(fig)

    # CP도 같은 색과 경과시간으로 비교하기 위함
    fig, ax = plt.subplots(figsize=(13, 4), layout="constrained")
    for index, (lot, label) in enumerate(zip(lot_ids, legend_labels)):
        color = COLORS[index % len(COLORS)]
        mean, low, high = means[lot], lows[lot], highs[lot]
        elapsed = (mean.index - lots.loc[lot, "시작"]).total_seconds() / 60
        ax.plot(elapsed, mean["CP-Z4"], color=color, linewidth=1, label=label)
        ax.fill_between(elapsed, low["CP-Z4"], high["CP-Z4"], color=color, alpha=0.08)
    ax.set_title(f"{grade} | CP-Z4 | {message}")
    ax.set_ylabel("CP-Z4 기록값 (단위 미확인)")
    ax.set_xlabel("첫 온도 관측 후 경과시간 (분) · 선: 1분 평균 · 옅은 색: 그 분의 최소~최대")
    ax.grid(alpha=0.15)
    ax.legend(loc="upper left", fontsize=8)
    output = FIGURES / "CP_기록값" / "강종별"
    output.mkdir(parents=True, exist_ok=True)
    fig.savefig(output / f"{grade}_CP_Z4.png", dpi=140)
    if grade == VIEW_GRADE:
        plt.show()
    plt.close(fig)


## 3. 그림을 읽고 기록할 내용

- 온도는 비슷한데 출력 수준이 다른 LOT가 있는지 보기.
- Z3·Z4·Z5가 함께 변하는지, 특정 구역만 변하는지 보기.
- CP 변화가 온도·출력 변화와 같은 때인지 보기. 같은 때라는 사실만으로 원인을 정하지 않음.
- 한 LOT에서만 보이는 모습과 같은 강종의 여러 LOT에서 반복되는 모습을 구분하기.

**이 그림은 팀에서 정한 7개 예측 대상의 특징을 설명하기 위한 것임. 7개가 다른 모든 채널보다 좋은 예측 대상이라는 사실을 증명하는 것은 아님.** 높은 출력이나 선의 차이를 곧바로 정상·이상 정답으로 만들지 않음.

038·114의 SPCEN 비교에서는 038의 높은 OP-Z4를 유지함. 058의 정지와 061의 재가동은 팀 확정 해석을 유지함. 단일 LOT 강종은 강종 특성과 해당 LOT의 운전 상태를 분리하기 어려움.
